In [1]:
# ============================================================
# Comparar_enlaces_ARIMA_h.py — v1.2  (celda de Jupyter, NIVEL HORARIO)
# MAE, RMSE, MAPE y WAPE por enlace de cada arquitectura horaria frente a ARIMA (T+1 h).
#   - DATASET = "abilene" o "brain" (rutas y TRIGGER de cada uno).
#   - Una tabla por métrica, ventana W (24, 168) y modo (A, B).
#   - Mismos instantes para todos los modelos (inner join con ARIMA).
#   - Una tabla por métrica, horizonte y modo: filas = enlaces, columnas = ARIMA y
#     Arch1–4; el mejor de cada enlace en negrita.
#   - Al final de cada tabla:
#       GLOBAL         → media de los enlaces (MAE, RMSE, MAPE); WAPE = Σ|e| / Σ real
#       % gana ARIMA   → % de enlaces en los que la arquitectura tiene menor métrica
#   - Formato: > 999 → 1.3e3 ; 99 … 999 → entero ; < 99 → un decimal.
#   - GUARDAR_CSV = True guarda cada tabla (valores sin formato) en DIR_SALIDA.
# ============================================================
import os
import numpy as np
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP
from IPython.display import display, HTML

DATASET = "abilene"          # "abilene" | "brain"

if DATASET == "abilene":
    ROOT_RESULTS = "Results_horas_abilene"
    DIR_BASE     = "Results_horas_abilene/results_baselines_1h"
    TRIGGER      = 0.001                               # todos los instantes
else:
    ROOT_RESULTS = "Results_horas"
    DIR_BASE     = "Results_horas/results_baselines_1h"
    TRIGGER      = 1000

COL_ARIMA   = lambda W: "ARIMA_t+1"                    # T+1 h

CARPETA     = "results_Arq{ID}_1h_{MODO}_W{H}"         # aquí {H} es la ventana W
COL_PRED    = "Prediccion"
IDS         = [1, 2, 3, 4]
HORIZONTES  = [24, 168]                                # ventanas W (horas)
MODOS       = ["A", "B"]
METRICAS    = ["MAE", "RMSE", "MAPE", "WAPE"]
GUARDAR_CSV = False
DIR_SALIDA  = f"Tablas_enlaces_h_{DATASET}"


def _red(x, dec):
    return Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-dec), rounding=ROUND_HALF_UP)


def val(v):
    if pd.isna(v):
        return "--"
    if v > 999:
        e = int(np.floor(np.log10(v)))
        m = _red(v / 10 ** e, 1)
        if m >= 10:
            m, e = _red(m / 10, 1), e + 1
        return f"{m}e{e}"
    if v >= 99:
        return str(int(_red(v, 0)))
    return f"{_red(v, 1)}"


def leer(ID, H, MODO):
    d = os.path.join(ROOT_RESULTS, CARPETA.format(ID=ID, H=H, MODO=MODO))
    if not os.path.isdir(d):
        print(f"   ⏭  Arch{ID} W{H} {MODO}: no existe {d}")
        return {}
    pref = "test_" + os.path.basename(d).removeprefix("results_") + "_"
    return {f[len(pref):-4]: os.path.join(d, f) for f in os.listdir(d)
            if f.startswith(pref) and f.endswith(".csv")}


def metricas(y, p):
    err = y - p
    m = y > 0
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean()),
            "MAPE": np.mean(np.abs(err[m]) / y[m]) * 100 if m.any() else np.nan,
            "WAPE": np.abs(err).sum() / y.sum() * 100}


def tabla(H):
    """Cada arquitectura con los enlaces que tenga. Para cada enlace y arquitectura,
    la arquitectura y ARIMA se evalúan en los mismos instantes (arquitectura ∩ ARIMA)."""
    arqs = {(ID, MODO): a for ID in IDS for MODO in MODOS if (a := leer(ID, H, MODO))}
    if not arqs:
        return None, None, None
    modelos = ["ARIMA"] + [f"Arch{ID}_{MODO}" for (ID, MODO) in arqs]
    filas = []
    se = {m: 0.0 for m in modelos}; sy = {m: 0.0 for m in modelos}
    gana = {m: [] for m in modelos[1:]}
    for e in sorted(set.union(*[set(a) for a in arqs.values()])):
        fb = os.path.join(DIR_BASE, f"test_results_{e}_baselines.csv")
        if not os.path.exists(fb):
            continue
        b = pd.read_csv(fb, parse_dates=["timestamp"]).set_index("timestamp")
        if COL_ARIMA(H) not in b.columns:
            raise SystemExit(f"❌ {fb} no tiene {COL_ARIMA(H)}; columnas: {list(b.columns)}")
        b = b[[COL_ARIMA(H)]].rename(columns={COL_ARIMA(H): "ARIMA"})
        fila, ts_arima = {"Link": e}, None
        for (ID, MODO), a in arqs.items():
            m = f"Arch{ID}_{MODO}"
            if e not in a:
                continue
            t = pd.read_csv(a[e], parse_dates=["timestamp"]).set_index("timestamp")
            j = t[["Realidad", COL_PRED]].rename(columns={COL_PRED: m}).join(b, how="inner").dropna()
            if TRIGGER is not None:
                j = j[j["Realidad"] > TRIGGER]
            if j.empty:
                continue
            y = j["Realidad"].values
            r, ra = metricas(y, j[m].values), metricas(y, j["ARIMA"].values)
            for met in METRICAS:
                fila[(met, m)] = r[met]
            gana[m].append({met: r[met] < ra[met] for met in METRICAS})
            se[m] += np.abs(y - j[m].values).sum(); sy[m] += y.sum()
            if ts_arima is None:                       # ARIMA de la fila: instantes de la 1.ª arquitectura
                ts_arima = j
        if ts_arima is None:
            continue
        y = ts_arima["Realidad"].values
        r = metricas(y, ts_arima["ARIMA"].values)
        for met in METRICAS:
            fila[(met, "ARIMA")] = r[met]
        se["ARIMA"] += np.abs(y - ts_arima["ARIMA"].values).sum(); sy["ARIMA"] += y.sum()
        filas.append(fila)
    if not filas:
        return None, None, None
    df = pd.DataFrame(filas)
    for m in modelos:
        for met in METRICAS:
            if (met, m) not in df:
                df[(met, m)] = np.nan
    glob = {met: {m: df[(met, m)].mean() for m in modelos} for met in ["MAE", "RMSE", "MAPE"]}
    glob["WAPE"] = {m: 100 * se[m] / sy[m] if sy[m] else np.nan for m in modelos}
    glob["n"] = {m: int(df[("MAE", m)].notna().sum()) for m in modelos}
    glob["gana"] = {m: {met: 100 * np.mean([g[met] for g in gana[m]]) if gana[m] else np.nan
                        for met in METRICAS} for m in modelos[1:]}
    return df, modelos, glob


def mostrar(df, modelos, glob, met, H):
    neg = lambda t, b: f"<b>{t}</b>" if b else t
    filas, crudo = [], []
    for _, r in df.iterrows():
        v = {m: r[(met, m)] for m in modelos}
        hay = {m: x for m, x in v.items() if pd.notna(x)}
        mejor = min(hay, key=hay.get) if hay else None
        filas.append({"Link": r["Link"], **{m: neg(val(v[m]), m == mejor) for m in modelos}})
        crudo.append({"Link": r["Link"], **v})
    g = glob[met]
    hay = {m: x for m, x in g.items() if pd.notna(x)}
    mejor = min(hay, key=hay.get)
    filas.append({"Link": "<b>GLOBAL</b>", **{m: neg(val(g[m]), m == mejor) for m in modelos}})
    filas.append({"Link": "nº enlaces", **{m: str(glob["n"][m]) for m in modelos}})
    filas.append({"Link": "% gana ARIMA", **{m: ("" if m == "ARIMA" else
                  f"{glob['gana'][m][met]:.1f}%") for m in modelos}})
    unidad = " (%)" if met in ("MAPE", "WAPE") else ""
    print(f"\n===== {DATASET} · {met}{unidad} por enlace · T+1 h · W={H} · modos A y B · "
          f"{len(df)} enlaces en total · negrita = mejor · -- = sin resultado =====")
    display(HTML(pd.DataFrame(filas).set_index("Link").to_html(escape=False)))
    if GUARDAR_CSV:
        os.makedirs(DIR_SALIDA, exist_ok=True)
        ruta = os.path.join(DIR_SALIDA, f"{met}_W{H}.csv")
        pd.DataFrame(crudo + [{"Link": "GLOBAL", **g}]).to_csv(ruta, index=False)


if not os.path.isdir(DIR_BASE):
    raise SystemExit(f"❌ No existe {DIR_BASE}")
for H in HORIZONTES:
    df, modelos, glob = tabla(H)
    if df is None:
        continue
    for met in METRICAS:
        mostrar(df, modelos, glob, met, H)


===== abilene · MAE por enlace · T+1 h · W=24 · modos A y B · 110 enlaces en total · negrita = mejor · -- = sin resultado =====


,ARIMA,Arch1_A,Arch1_B,Arch2_A,Arch2_B,Arch3_A,Arch3_B,Arch4_A,Arch4_B
Link,,,,,,,,,
ATLA_CHIN,2.8,2.5,2.5,2.5,2.5,2.5,2.6,2.5,3.0
ATLA_DNVR,1.4,1.3,1.3,1.2,1.3,1.2,1.2,1.3,1.3
ATLA_HSTN,7.1,7.2,7.6,7.2,8.2,7.1,8.5,7.2,8.1
ATLA_IPLS,2.6,2.4,2.4,2.4,2.4,2.3,2.4,2.5,2.7
ATLA_KSCY,1.3,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.3
ATLA_LOSA,2.9,2.7,2.8,2.8,2.8,2.7,2.7,2.8,2.7
ATLA_NYCM,1.9,1.7,1.7,1.7,1.7,1.8,1.8,1.8,1.7
ATLA_SNVA,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.5
ATLA_STTL,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9



===== abilene · RMSE por enlace · T+1 h · W=24 · modos A y B · 110 enlaces en total · negrita = mejor · -- = sin resultado =====


,ARIMA,Arch1_A,Arch1_B,Arch2_A,Arch2_B,Arch3_A,Arch3_B,Arch4_A,Arch4_B
Link,,,,,,,,,
ATLA_CHIN,3.8,3.5,3.6,3.5,3.6,3.5,3.7,3.6,4.3
ATLA_DNVR,2.4,2.2,2.3,2.2,2.3,2.2,2.2,2.2,2.2
ATLA_HSTN,11.1,10.8,11.5,10.9,12.3,10.8,12.8,10.7,12.3
ATLA_IPLS,3.7,3.5,3.4,3.5,3.5,3.4,3.5,3.6,3.9
ATLA_KSCY,2.2,1.9,1.9,1.9,1.9,1.9,1.9,1.9,2.1
ATLA_LOSA,3.8,3.6,3.7,3.7,3.6,3.7,3.6,3.8,3.6
ATLA_NYCM,2.8,2.6,2.5,2.6,2.5,2.7,2.6,2.7,2.6
ATLA_SNVA,0.8,0.8,0.8,0.8,0.9,0.8,0.8,0.8,1.0
ATLA_STTL,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3



===== abilene · MAPE (%) por enlace · T+1 h · W=24 · modos A y B · 110 enlaces en total · negrita = mejor · -- = sin resultado =====


,ARIMA,Arch1_A,Arch1_B,Arch2_A,Arch2_B,Arch3_A,Arch3_B,Arch4_A,Arch4_B
Link,,,,,,,,,
ATLA_CHIN,29.0,26.0,26.4,25.9,27.0,25.7,27.0,25.9,32.1
ATLA_DNVR,21.2,19.7,19.0,18.9,19.0,18.6,19.3,19.1,20.2
ATLA_HSTN,87.9,88.5,87.0,82.2,95.2,83.1,93.2,95.7,88.4
ATLA_IPLS,15.5,14.4,14.5,14.5,14.4,13.8,14.2,14.9,16.6
ATLA_KSCY,62.3,44.2,51.1,42.1,45.4,42.1,47.0,41.6,58.7
ATLA_LOSA,20.1,18.5,21.2,18.8,20.4,18.2,19.5,19.0,18.8
ATLA_NYCM,27.5,24.0,24.4,24.1,25.0,25.1,24.5,24.7,24.3
ATLA_SNVA,96.2,86.4,111,86.5,100,82.2,98.8,84.3,113
ATLA_STTL,18.2,16.9,18.0,16.9,17.8,16.9,17.1,17.1,18.8



===== abilene · WAPE (%) por enlace · T+1 h · W=24 · modos A y B · 110 enlaces en total · negrita = mejor · -- = sin resultado =====


,ARIMA,Arch1_A,Arch1_B,Arch2_A,Arch2_B,Arch3_A,Arch3_B,Arch4_A,Arch4_B
Link,,,,,,,,,
ATLA_CHIN,26.6,24.3,24.5,24.2,24.5,24.2,24.9,24.3,29.1
ATLA_DNVR,22.8,21.3,21.1,20.8,21.3,20.6,20.8,21.3,21.6
ATLA_HSTN,45.2,45.8,48.7,45.8,52.2,45.5,54.3,46.2,51.8
ATLA_IPLS,16.1,14.9,14.9,15.1,15.0,14.6,14.9,15.4,17.1
ATLA_KSCY,45.6,37.2,39.3,37.1,37.6,37.0,37.8,37.1,43.4
ATLA_LOSA,19.0,17.9,18.5,18.2,18.4,17.8,17.9,18.4,17.4
ATLA_NYCM,25.7,23.0,22.8,23.1,22.9,23.8,23.9,23.9,23.1
ATLA_SNVA,47.1,42.6,46.7,42.5,47.7,41.6,46.5,43.6,53.6
ATLA_STTL,17.4,16.3,17.5,16.3,17.2,16.2,16.5,16.2,17.5


   ⏭  Arch1 W168 A: no existe Results_horas_abilene/results_Arq1_1h_A_W168
   ⏭  Arch1 W168 B: no existe Results_horas_abilene/results_Arq1_1h_B_W168
   ⏭  Arch2 W168 A: no existe Results_horas_abilene/results_Arq2_1h_A_W168
   ⏭  Arch2 W168 B: no existe Results_horas_abilene/results_Arq2_1h_B_W168
   ⏭  Arch3 W168 A: no existe Results_horas_abilene/results_Arq3_1h_A_W168
   ⏭  Arch3 W168 B: no existe Results_horas_abilene/results_Arq3_1h_B_W168
   ⏭  Arch4 W168 A: no existe Results_horas_abilene/results_Arq4_1h_A_W168
   ⏭  Arch4 W168 B: no existe Results_horas_abilene/results_Arq4_1h_B_W168
